In [ ]:
#Notbook to run ISM for a given bed file in

%load_ext autoreload
%autoreload 2

import os
import sys
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy import stats

# Make the package in ../src importable without installing it.
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from element_epistasis import paths
from element_epistasis.ism import load_ism_dict

paths.ensure_output_dirs()

# True regenerates the AlphaGenome predictions, which needs an API key, the
# hg38 reference and several hours. False loads them from data/processed.
RUN_PREDICTIONS = False
from element_epistasis.GLM_fit import *


import element_epistasis.enhancer_analysis as ea
import element_epistasis.enhancer_plots as ep
import element_epistasis.tf_cooperativity as tc

import importlib
importlib.reload(ea)
importlib.reload(ep)
importlib.reload(tc)

In [ ]:
# The key is read from ALPHAGENOME_API_KEY or ~/.alphagenome_api_key.
api_key = paths.alphagenome_api_key()
model = dna_client.create(api_key) if api_key else None
if RUN_PREDICTIONS and model is None:
    raise RuntimeError("RUN_PREDICTIONS needs an AlphaGenome API key")


In [ ]:
#Set AlphaGenome Context length
SEQ_LEN = dna_client.SEQUENCE_LENGTH_1MB

In [ ]:
enhancers = pd.read_csv(paths.RAW / "enhancers.bed", sep = "\t", header=None, names=["chrom", "start", "end", "name"])
def extract_genes(name_string):
    return [part.split('_')[0] for part in name_string.split(':')]
def extract_enhancers(name_string):
    return [part for part in name_string.split(':')]

enhancers['target_gene'] = enhancers['name'].apply(extract_genes)
enhancers['enhancer_name'] = enhancers['name'].apply(extract_enhancers)

all_TFs = pd.read_csv(paths.RAW / "tfs.tsv", sep = "\t", header=None, names=["TF"])
all_TFs = all_TFs.drop_duplicates().TF.to_list()

enhancers =  enhancers.explode(['target_gene', 'enhancer_name'], ignore_index=True)
enhancers

In [ ]:
exp = pd.read_csv(paths.RAW / "enhancer_interactions_exp.csv")

t_gene = []
for r in exp.itertuples():
    if pd.isna(r.response_gene):
        t_gene.append(r.pair_name.split("_")[0])
    else:
        t_gene.append(r.response_gene)

exp["response_gene"] = t_gene

exp.head(5)

In [ ]:
if model is not None:
    # selecting relvant tracks from track metadata
    track_metadata = model.output_metadata(
        dna_client.Organism.HOMO_SAPIENS
    ).concatenate()

    K562_metadata = track_metadata[(track_metadata["biosample_name"].str.contains("K562", case=False, na=False))&
                   ((track_metadata["output_type"] == dna_client.OutputType.ATAC) | (track_metadata["output_type"] == dna_client.OutputType.CHIP_TF))
                  ].dropna(how="all", axis = 1)

    K562_curie = K562_metadata.ontology_curie.unique().tolist()
    K562_sample = K562_metadata.biosample_name.unique().tolist()

    print("curie:", K562_curie)
    print("samples:", K562_sample)
else:
    # No API key: use the curie the stored predictions were generated with.
    K562_curie = [paths.K562_CURIE]
    print("curie:", K562_curie, "(fallback, no API key)")


In [ ]:
from alphagenome_research.model.variant_scoring.gene_mask_extractor import (
    GeneMaskExtractor, GeneMaskType, GeneQueryType
)

# GENCODE v46, the annotation AlphaGenome uses; cached in data/reference.
gtf_df = paths.load_gtf()

gtf_df = gtf_df[gtf_df.gene_type == "protein_coding"]

In [ ]:
def get_gene_coord(gene):
    ret = gtf_df[(gtf_df.Feature == "gene") & (gtf_df.gene_name == gene)]
    if ret.empty:
        return None, None
    return ret.Start.values[0], ret.End.values[0]

enhancers['gene_start'], enhancers['gene_end'] = zip(*enhancers['target_gene'].apply(get_gene_coord))

In [ ]:
enhancers['iv_min_start'] = (
    enhancers[['start', 'gene_start']]
    .min(axis=1)
    .groupby(enhancers['target_gene'])
    .transform('min')
)

enhancers['iv_max_end'] = (
    enhancers[['end', 'gene_end']]
    .max(axis=1)
    .groupby(enhancers['target_gene'])
    .transform('max')
)

assert (enhancers.iv_max_end -enhancers.iv_min_start <= SEQ_LEN).all(), "All Enhancer gene pairs must be in the prediction input interval"

In [ ]:
enhancers

In [ ]:
#check how well the fit into the interval
plt.hist(enhancers.iv_max_end -enhancers.iv_min_start, bins = len(enhancers))
plt.xlim(0, SEQ_LEN)
plt.xlabel("Minimum Interval Distance")
plt.show()

In [ ]:
iv_center =  enhancers.iv_min_start + (enhancers.iv_max_end - enhancers.iv_min_start) // 2
iv_start = iv_center - SEQ_LEN // 2
#handle chrom start overlap
iv_start[iv_start < 60_000] = 60_000 #60_000 is the first base that is not N after a chrom start on hg38 so first start if the actiual sequence
iv_end = iv_start + SEQ_LEN

assert (iv_end - iv_start == SEQ_LEN).all()

In [ ]:
enhancers["iv_start"] = iv_start
enhancers["iv_end"] = iv_end
enhancers["interval"] = [genome.Interval(chromosome=row.chrom, start=row.iv_start, end=row.iv_end).resize(SEQ_LEN) for row in enhancers.itertuples()]

In [ ]:
# Chromosome sizes are only needed to check the prediction windows.
if paths.CHROM_SIZES.exists():
    chrom_size = pd.read_csv(
        paths.CHROM_SIZES,
        sep="\t",
        names=["chrom", "end"]
    )
    chrom_size_dict = dict(zip(chrom_size["chrom"], chrom_size["end"]))


In [ ]:
# Chromosome sizes are only needed to check the prediction windows.
if paths.CHROM_SIZES.exists():
    assert len([idx for idx, row in enhancers.iterrows() if row.iv_end > chrom_size_dict[row.chrom] - 60_000]) == 0, "Some interval resach chorm ends."


In [ ]:
if RUN_PREDICTIONS:
    from pyfaidx import Fasta

    hg38_path = str(paths.HG38_FASTA)
    hg38 = Fasta(hg38_path)

    enhancers["seq"] = [str(hg38[iv.chromosome][iv.start:iv.end]) for iv in enhancers.interval]


In [ ]:
if RUN_PREDICTIONS:
    import torch
    from tangermeme.utils import one_hot_encode

    seq_1hot = torch.stack([one_hot_encode(seq) for seq in enhancers.seq])
    seq_1hot.shape


In [ ]:
if RUN_PREDICTIONS:
    #we have some N in the input sequnce so we feilter them out shuffle without them an dpad to legnth agian
    x = seq_1hot.sum(dim=1)
    valid_mask = (x == 1)
    valid_cols_mask = valid_mask.all(dim=0)
    filtered_1hot = seq_1hot[:, :, valid_cols_mask]
    print("Filtered shape:", filtered_1hot.shape)


In [ ]:
if RUN_PREDICTIONS:
    from tangermeme.ersatz import dinucleotide_shuffle
    dishuff = dinucleotide_shuffle(filtered_1hot, n=10, random_state=42)

    #pass deleted bases with just the reamian random slice
    dishuff_pad = dishuff[:, :, :, :SEQ_LEN-filtered_1hot.shape[2]]
    dishuff = torch.cat([dishuff, dishuff_pad], dim=3)
    dishuff.shape


In [ ]:
from tangermeme.utils import one_hot_encode


"""
Generate all seqlet scramble combinations per enhancer.

For each enhancer with n seqlets, produces 1 + (2^n - 1) * 10 sequences:
  - 1  row : all seqlets unscrambled  → original enhancer sequence
  - 10 rows: per non-trivial combo    → each of the 10 dinucleotide shuffles,
               with the scrambled-seqlet windows replaced from that shuffle

The `dishuff` tensor must already be loaded in scope:
    dishuff.shape == [95, 10, 4, 313]  (n_enhancers, n_shuffles, 4, seq_len)
"""

import itertools
import numpy as np
import pandas as pd
import torch

# ACGT order assumed (standard one-hot for genomics)
_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def onehot_to_seq(onehot: torch.Tensor) -> str:
    """
    Convert a one-hot tensor of shape [4, L] to an uppercase nucleotide string.
    Handles ties by taking argmax (first max index).
    """
    indices = onehot.argmax(dim=0).numpy()   # shape [L]
    return "".join(_IDX_TO_NUC[indices])


def splice_scramble(
    original_seq: str,
    scramble_windows: list[tuple[int, int]],
    donor_seq: str,
) -> str:
    """
    Replace every (start, end) window in `original_seq` with the same window
    from `donor_seq`.  start/end are 0-based, end exclusive (Python slicing).
    """
    seq = list(original_seq)
    for start, end in scramble_windows:
        seq[start:end] = list(donor_seq[start:end])
    return "".join(seq)


def generate_combination_df(
    motif_df: pd.DataFrame,
    dishuff: torch.Tensor,
    enhancer_col: str = "name",
    seq_col: str = "seq",
    seq_idx_col: str = "seq_idx",
    start_col: str = "start_seq",
    end_col: str = "end_seq",
    seqlet_name_col: str = "example_idx",   # human-readable seqlet label
) -> pd.DataFrame:
    """
    Parameters
    ----------
    motif_df    : DataFrame as described (one row per seqlet)
    dishuff     : Tensor [n_enhancers, 10, 4, seq_len], dinucleotide shuffles
    *_col       : column name overrides

    Returns
    -------
    pd.DataFrame with columns:
        name, seq_idx, combo_id, combo_label,
        scrambled_seqlets, unscrambled_seqlets,
        n_scrambled, shuffle_idx, sequence
    """
    n_shuffles = dishuff.shape[1]
    rows = []

    for enhancer, grp in motif_df.groupby(enhancer_col):
        grp = grp.reset_index(drop=True)

        original_seq = grp[seq_col].iloc[0].upper()
        seq_idx      = int(grp[seq_idx_col].iloc[0])

        # Seqlets for this enhancer, sorted by start position
        grp_sorted = grp.sort_values(start_col).reset_index(drop=True)
        n_seqlets  = len(grp_sorted)

        # Pre-convert all 10 shuffled tensors to strings once
        enhancer_shuffles = dishuff[seq_idx]          # [10, 4, L]
        shuffle_seqs = [
            onehot_to_seq(enhancer_shuffles[i])
            for i in range(n_shuffles)
        ]

        # Seqlet metadata lists (aligned with grp_sorted rows)
        seqlet_labels  = grp_sorted[seqlet_name_col].tolist()
        seqlet_windows = list(
            zip(grp_sorted[start_col].tolist(), grp_sorted[end_col].tolist())
        )

        # Iterate over all 2^n bitmask combinations
        # bit i == 1  →  seqlet i is SCRAMBLED
        for combo_id in range(2 ** n_seqlets):
            bits = [(combo_id >> i) & 1 for i in range(n_seqlets)]

            scrambled_idx   = [i for i, b in enumerate(bits) if b]
            unscrambled_idx = [i for i, b in enumerate(bits) if not b]

            scrambled_labels   = [seqlet_labels[i]  for i in scrambled_idx]
            unscrambled_labels = [seqlet_labels[i]  for i in unscrambled_idx]
            scrambled_windows  = [seqlet_windows[i] for i in scrambled_idx]

            # Human-readable label, e.g. "scr[GATA2,STAT5A]_unscr[MYB73]"
            combo_label = (
                f"scr[{','.join(str(s) for s in scrambled_labels)}]"
                f"_unscr[{','.join(str(s) for s in unscrambled_labels)}]"
            )

            if combo_id == 0:
                # ── all unscrambled: use original sequence, no shuffle ──
                rows.append(
                    dict(
                        name       = enhancer,
                        seq_idx             = seq_idx,
                        combo_id            = combo_id,
                        combo_label         = combo_label,
                        scrambled_seqlets   = scrambled_labels,
                        unscrambled_seqlets = unscrambled_labels,
                        n_scrambled         = 0,
                        shuffle_idx         = None,
                        sequence            = original_seq,
                    )
                )
            else:
                # ── scrambled combo: one row per shuffle ──
                for shuf_i, donor in enumerate(shuffle_seqs):
                    scrambled_seq = splice_scramble(
                        original_seq, scrambled_windows, donor
                    )
                    rows.append(
                        dict(
                            name       = enhancer,
                            seq_idx             = seq_idx,
                            combo_id            = combo_id,
                            combo_label         = combo_label,
                            scrambled_seqlets   = scrambled_labels,
                            unscrambled_seqlets = unscrambled_labels,
                            n_scrambled         = len(scrambled_idx),
                            shuffle_idx         = shuf_i,
                            sequence            = scrambled_seq,
                        )
                    )

    result = pd.DataFrame(rows)

    # Tidy column order
    col_order = [
        "name", "seq_idx", "combo_id", "combo_label",
        "n_scrambled", "scrambled_seqlets", "unscrambled_seqlets",
        "shuffle_idx", "sequence",
    ]
    return result[col_order]

In [ ]:
enhancers['start'] = enhancers['start'] - enhancers['iv_start']
enhancers['end'] = enhancers['end'] - enhancers['iv_start']

In [ ]:
enhancers['seq_idx'] = pd.factorize(enhancers['target_gene'])[0]
enhancers

In [ ]:
# motif_df: pd.DataFrame,
#     dishuff: torch.Tensor,
#     enhancer_col: str = "name",
#     seq_col: str = "seq",
#     seq_idx_col: str = "seq_idx",
#     start_col: str = "start_seq",
#     end_col: str = "end_seq",
#     seqlet_name_col: str = "example_idx",   # human-readable seqlet label

In [ ]:
if RUN_PREDICTIONS:
    out_df = generate_combination_df(enhancers, dishuff, seq_col="seq", seq_idx_col="seq_idx", start_col = "start", end_col="end", enhancer_col="target_gene", seqlet_name_col="enhancer_name")


In [ ]:
if RUN_PREDICTIONS:
    out_df


In [ ]:
if RUN_PREDICTIONS:
    #santiy check scramble for beginnign of first enhancer and first two scrams
    [print(f"{seq[515886-10:515886]} {seq[515886:515886+10]}") for seq in out_df.iloc[:3, -1]]


In [ ]:
if RUN_PREDICTIONS:
    print(f"\nTotal rows: {len(out_df)}")
    print(f"Unique enhancers: {out_df['name'].nunique()}")

    # Quick sanity check: all sequences same length per enhancer
    seq_lens = out_df.groupby("name")["sequence"].apply(
        lambda s: s.str.len().unique()
    )
    assert all(len(v) == 1 for v in seq_lens), "Sequence length mismatch!"
    print("All sequences per enhancer have consistent length.")


# run prediction

In [ ]:
exon_extractor = GeneMaskExtractor(
    gtf=gtf_df,
    gene_mask_type=GeneMaskType.EXONS,
    gene_query_type=GeneQueryType.INTERVAL_CONTAINED,
    filter_protein_coding=False
)

In [ ]:
def mean_of_lists(series):
    """Element-wise mean across a series of lists."""
    parsed = [
        ast.literal_eval(v) if isinstance(v, str) else v
        for v in series
    ]
    return np.array(parsed).mean(axis=0).tolist()

def sd_of_lists(series):
    """Element-wise sd across a series of lists."""
    parsed = [
        ast.literal_eval(v) if isinstance(v, str) else v
        for v in series
    ]
    return np.array(parsed).std(axis=0).tolist()

In [ ]:
if RUN_PREDICTIONS:
    import time
    import grpc
    df_list, df_list_agg = [], []

    # Config for API robustness
    MAX_RETRIES = 5
    MAX_WORKERS = 6 #go upt to 5 but might fail if high demand

    for name in out_df.name.unique():
        print(f"\nAnalysing enhancer: {name}")
        enh_scram = out_df[out_df.name == name]
        edata = enhancers[enhancers.target_gene == name]

        input_interval = edata.interval.values[0]
        model_inputs = enh_scram.sequence.to_list()

        out_pred = None   
        # Reset to original max workers for this new batch
        current_workers = MAX_WORKERS 

        # try connecting to api and retry if failed to avoid entire loop crashes
        for attempt in range(MAX_RETRIES):
            try:
                out_pred = model.predict_sequences(
                    sequences=model_inputs,
                    requested_outputs=[dna_client.OutputType["ATAC"], dna_client.OutputType["RNA_SEQ"]],
                    ontology_terms=K562_curie,
                    max_workers=current_workers,  # Use the dynamically adjusting worker count
                    intervals=[input_interval] * len(model_inputs)
                )
                break  # Break the retry loop and continue pipeline
            
            except Exception as e: # Catches gRPC and API wrapper errors
                print(f"     [!] API Error on attempt {attempt + 1}: {e}")
                if attempt < MAX_RETRIES - 1:
                    # Drop the worker count to reduce server load
                    if current_workers > 2:
                        current_workers -= 2
                    else:
                        current_workersent_workes = 1
                    print(f"     [*] Decreasing max_workers to {current_workers}...")
                    wait_time = 15 * (attempt + 1) # 15s wait before retrying
                    print(f"     [*] Sleeping {wait_time}s before retrying...")
                    time.sleep(wait_time)
                else:
                    print("     [!!!] Max retries reached. Exiting pipeline.")
                    raise e

        # get gene masks once per interval
        gene_mask, annot = exon_extractor.extract(interval=out_pred[0].rna_seq.interval)
        is_target = annot.gene_name == name
        annot = annot.loc[is_target]
        gene_mask = gene_mask[:, is_target]

        # iterate over shuffles
        RNA_counts, ATAC_counts, enhs = [], [], []
        for seq_out in out_pred:

            # get enhancer ATAC mask aggregation and sum atac over enhancer
            enh_name, enh_counts = [], []
            for e in edata.itertuples(): 
                enh_name.append(e.enhancer_name)
                mask = np.zeros(SEQ_LEN, dtype=bool)
                mask[e.start:e.end] = True
                enh_counts.append(seq_out.atac.values.mean(axis=1)[mask].sum())

            enhs.append(enh_name)
            ATAC_counts.append(enh_counts)

            # sum gene counts       
            strand = annot.strand.values[0]
            stranded_track = seq_out.rna_seq.metadata.strand.isin([strand, "."])
            gene_counts = seq_out.rna_seq.values[:, stranded_track].mean(axis=1)[gene_mask[:, 0]].mean()
            RNA_counts.append(gene_counts)

        del out_pred
        enh_scram["RNA_counts"] = RNA_counts
        enh_scram["ATAC_counts"] = ATAC_counts
        enh_scram["target_enhancer"] = enhs
    
        # aggregate the shuffled ref sequences by avg
        enh_scram_agg = (
            enh_scram
            .groupby(
                ["name", "seq_idx", "combo_id", "combo_label", "n_scrambled"],
                observed=True,
            )
            .agg(
                ATAC_counts         = ("ATAC_counts",         mean_of_lists),
                ATAC_counts_sd      = ("ATAC_counts",         sd_of_lists),
                RNA_counts          = ("RNA_counts",          "mean"),
                RNA_counts_sd       = ("RNA_counts",          "std"),
                target_enhancer     = ("target_enhancer",     "first"),
                scrambled_seqlets   = ("scrambled_seqlets",   "first"),
                unscrambled_seqlets = ("unscrambled_seqlets", "first"),
            )
            .reset_index()
        )
        df_list.append(enh_scram)
        df_list_agg.append(enh_scram_agg)
        del enh_scram, enh_scram_agg


In [ ]:
if RUN_PREDICTIONS:
    scram_effects = pd.concat(df_list)
    scram_effects = scram_effects.reset_index(drop=True)

    scram_effects_agg = pd.concat(df_list_agg)
    scram_effects_agg = scram_effects_agg.reset_index(drop=True)


In [ ]:
if RUN_PREDICTIONS:
    scram_effects.head(5)


In [ ]:
if RUN_PREDICTIONS:
    scram_effects_agg.head(5)


In [ ]:
if RUN_PREDICTIONS:
    scram_effects.drop(columns="sequence").to_parquet(paths.PROCESSED / "enhancer_insilico_KO.parquet", engine='pyarrow', index=False)
    scram_effects_agg.to_parquet(paths.PROCESSED / "enhancer_insilico_KO_agg.parquet", engine='pyarrow', index=False)


# Analysis

In [ ]:
scram_effects = pd.read_parquet(paths.PROCESSED / "enhancer_insilico_KO.parquet", engine='pyarrow')
scram_effects_agg = pd.read_parquet(paths.PROCESSED / "enhancer_insilico_KO_agg.parquet", engine='pyarrow')

## fit Linear model
we use a log linked linear model to fit a negbionm ditribution or a gamma to get the interaction terms

In [ ]:
scram_effects

In [ ]:
#compare distrubtion for one gene
for fam in ["nb", "gamma"]:
    res = fit_interaction_glm(scram_effects, "ANXA1",
                              family=fam, max_order=2, verbose=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_coef_forest(res, ax=ax)
    plt.tight_layout(); plt.show()

**From this we can clearly see that rounding to fit NB does not make sense here so we just stick with the Gamma**

In [ ]:
# all genes pairwise interactions
interactions_gm = fit_all_genes(scram_effects, family="gamma", max_order=2)

In [ ]:
#2
interactions_gm_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in interactions_gm.items()])
#3
interactions_gm_3 = fit_all_genes(scram_effects, family="gamma", max_order=3)
interactions_gm_3_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in interactions_gm_3.items()])

#we can see taht they are the same but with alrge se and ci for more fitted terms
pd.merge(interactions_gm_3_df, interactions_gm_df, on = ["gene", "family", "term"], how = "inner", suffixes=("_3", "_2"))

## compare with in vitro

In [ ]:
inter_betas = pd.read_csv(paths.RAW / "pair_interaction_adj.csv")
inter_betas = inter_betas.groupby(["enhA", "enhB"], observed = True).agg({"interaction_beta": "mean"}).reset_index() #avg beta from experimetns


single_betas = pd.read_csv(paths.RAW / "enhancer_betas.csv")
single_betas = single_betas.groupby("enhancer").agg({"beta": "mean"}).reset_index() #avg beta from experimetns
single_betas = single_betas.rename(columns = {"beta": "interaction_beta"})

In [ ]:
in_silico_single = interactions_gm_df[~interactions_gm_df['term'].str.contains(":|intercept", na=False)].copy()
in_silico_single["enhancer"] = in_silico_single["gene"] + "_" + in_silico_single["term"]
in_silico_single

In [ ]:
in_silico_inter = interactions_gm_df[interactions_gm_df['term'].str.contains(":", na=False)].copy()
in_silico_inter[['enhA', 'enhB']] = [(g+"_"+a, g+"_"+b) for g, (a, b) in zip(in_silico_inter['gene'], in_silico_inter['term'].str.split(":"))]
in_silico_inter

### single beta comparison

In [ ]:
single_comp = pd.merge(in_silico_single, single_betas, on = "enhancer", how = "inner")
#single_comp = single_comp[single_comp.pval <= 0.05]
single_comp.head(3)

### pariwise comparison

In [ ]:
pair_comp = pd.merge(in_silico_inter, inter_betas, on = ["enhA", "enhB"], how = "inner")
#pair_comp = pair_comp[pair_comp.pval <= 0.05]
pair_comp

# ATAC analysis

In [ ]:
atac_results = fit_interaction_glm_atac_all_genes(scram_effects, family="gamma", max_order=2)
rna_result   = fit_interaction_glm(scram_effects, gene="ANXA1", family="gamma", max_order=2)

fig = plot_coef_forest_grid(rna_result=rna_result, atac_result=atac_results["ANXA1"], crispr_single = single_betas, crispr_inter = inter_betas)
plt.show()

In [ ]:
atac_results_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in atac_results.items()])
atac_results_df = atac_results_df[atac_results_df['term'].str.contains(":", na=False)]
atac_results_df[['enhA', 'enhB']] = [(g+"_"+a, g+"_"+b) for g, (a, b) in zip(atac_results_df['gene'], atac_results_df['term'].str.split(":"))]

#aggreagte
def absmax(series: pd.Series) -> float:
    return series.loc[series.abs().idxmax()]
atac_results_df = atac_results_df.groupby(["gene","enhA","enhB"], observed=True).agg({"coef": absmax, "pval": "min"}).reset_index()

atac_comp = pd.merge(atac_results_df, inter_betas, on = ["enhA", "enhB"], how = "inner")
atac_comp.head(3)

# New analysis

In [ ]:
mains, inters = ea.split_coef_table(interactions_gm_df)
mains  = tc.add_bh_correction(mains)

inters = tc.add_bh_correction(inters)
inters = ea.attach_main_effects(inters, interactions_gm_df)   # NOTE: unsplit table
inters = ea.annotate_pair_distance(inters, enhancers)

In [ ]:
#save these for the cross enahcer TF TF interactions
mains.to_csv(paths.PROCESSED / "enhancer_insilico_single_KO_main_effects.csv",
             header=True, index=False)
inters.to_csv(paths.PROCESSED / "enhancer_insilico_KO_inters.csv",
             header=True, index=False)

In [ ]:
atac_inters = ea.normalise_interaction_table(atac_results_df)
atac_inters = tc.add_bh_correction(atac_inters)

coop_rna = ea.build_coop_table(scram_effects, n_scrambled_col = "n_scrambled", value_col="RNA_counts")

epi_rna  = ea.build_pairwise_epistasis(scram_effects, n_scrambled_col = "n_scrambled", value_col="RNA_counts")
#epi_atac  = ea.build_pairwise_epistasis(scram_effects, n_scrambled_col = "n_scrambled", value_col="ATAC_counts")


summaries = {
    "single enhancers": ea.summarise_validation(
        single_comp, insilico_p_col="pval"),
    "enhancer pairs": ea.summarise_validation(
        pair_comp, insilico_p_col="pval"),
}

In [ ]:
ep.plot_validation_summary

In [ ]:
fig = ep.make_main_figure(inters, coop_rna, single_comp, pair_comp, summaries)
plt.show()

In [ ]:
#stratifiying to significant aris in the pair enahcer crispiri benchmark didi not iporve results
fig, ax = plt.subplots(1,1)
sig_pair_comp = tc.add_bh_correction(pair_comp)
ep.plot_insilico_vs_measured(sig_pair_comp[sig_pair_comp.significant], ax=ax, level="pair")
plt.show()

In [ ]:
interactions_gm_4 = fit_all_genes(scram_effects, family="gamma", max_order=4)

interactions_nb = fit_all_genes(scram_effects, family="nb", max_order=2)
interactions_nb_3 = fit_all_genes(scram_effects, family="nb", max_order=3)
interactions_nb_4 = fit_all_genes(scram_effects, family="nb", max_order=4)


comparison = ea.build_model_comparison({
    "gamma, order 2": interactions_gm,
    "gamma, order 3": interactions_gm_3,
    "gamma, order 4": interactions_gm_4,
    "NB, order 2":    interactions_nb,
    "NB, order 3":    interactions_nb_3,
    "NB, order 4":    interactions_nb_4,
}, verbose = False)

In [ ]:
fig = ep.make_supplementary_figure(inters, inters, atac_inters, comparison=comparison, figsize=(10, 10))
plt.show()

In [ ]:
# Summed: one row per gene
summed   = ea.aggregate_locus_values(scram_effects, "ATAC_counts", how="mean")
coop_atac_sum = ea.build_coop_table(summed, n_scrambled_col="n_scrambled",
                               value_col="ATAC_counts")

# Expanded: one row per locus
long     = ea.expand_locus_values(scram_effects, "ATAC_counts")
coop_atac_loc = ea.build_coop_table(long, name_col="locus",
                               n_scrambled_col="n_scrambled", value_col="ATAC_counts")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
ep.plot_additivity(coop_atac_sum, ax=axes[0], readout="ATAC, mean")
ep.plot_additivity(coop_atac_loc, ax=axes[1], readout="ATAC, per locus")
ep.plot_coop_agreement(coop_atac_sum, coop_atac_loc, ax=axes[2])
plt.show()

In [ ]:
importlib.reload(ep)
importlib.reload(ea)

In [ ]:
fig, ax = plt.subplots(1,1,figsize = (18,18))
ep.plot_global_interdependency(atac_results, ax = ax, show_only_sig=True,
                               hatch_nonsignificant = True, clip_quantile=0.3,
                               ylabel_pad = 55.0,
                               gene_label_x = -0.056,)
plt.show()


In [ ]:
interdependency_atac = ea.build_interdependency(long, value_col="ATAC_counts")

trans_gene  = ea.summarise_trans_effects(interdependency_atac, by="gene")
trans_locus = ea.summarise_trans_effects(interdependency_atac, by="perturbed")

pair_trans = ea.summarise_pair_trans(interdependency_atac)
pair_comp_2  = ea.join_pair_trans(pair_comp, pair_trans)

ep.make_mechanism_figure(trans_gene,
                         trans_locus, coop_atac_sum,
                         coop_atac_loc, coop_rna, pair_comp=pair_comp_2, figsize = (10, 10))
plt.show()

In [ ]:
fig = ep.make_gene_figure("PRKAR2B", interactions_gm["PRKAR2B"], epi_rna, scram_effects_agg)
plt.show()

In [ ]:
fig = ep.make_gene_figure("DDIT4", interactions_gm["DDIT4"], epi_rna, scram_effects_agg)
plt.show()

# Save a main sumary file

In [ ]:
from element_epistasis.pipeline import run_all

In [ ]:
tables = run_all(
    enh_coef=interactions_gm_df,
    enh_atac_coef=atac_results_df,
    enh_scram=scram_effects,
    enhancers=enhancers,
    single_comp=single_comp,
    pair_comp=pair_comp,
    model_results={"gamma, order 2": interactions_gm,
                   "gamma, order 3": interactions_gm_3},
    summary_path=paths.SUMMARIES / "enhancer_summary.txt",
)